# Runbook 5.4.c: Digital signatures

**Skill:** Mitigate Risk · **Maps to:** 5.4.B · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** A software vendor wants to let users verify that a downloaded update really came from them and
was not modified in transit. Rather than signing the whole update file directly with RSA (slow,
and structurally risky per atom.md), implement the standard "hash, then sign" digital signature
scheme: hash the message with SHA-256, reduce the digest mod n, and apply the RSA private-key
operation to that digest, not the raw message.

**Lesson objective.** Explain how a digital signature is created and verified using an asymmetric key pair and a cryptographic hash, and articulate why a valid signature provides authenticity, integrity, and non-repudiation in a way that a shared-secret MAC (5.3.c) cannot.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
Using the textbook-scale key pair `p=61, q=53 -> n=3233`, `e=17`, `d=2753`:
```
sign(b"pay Bob $50", 2753, 3233)        -> some int in [0, 3233)
verify(b"pay Bob $50", <that signature>, 17, 3233)   -> True
verify(b"pay Bob $950", <that signature>, 17, 3233)  -> False   # message tampered after signing
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only (`hashlib` only, no external crypto packages).
- [ ] Correctness is checked in the tests against an independent oracle that recomputes the same
- [ ] Must pass all cases in the tests (visible + hidden), including: a tampered message failing
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry points
```python
def sign(message: bytes, d: int, n: int) -> int
def verify(message: bytes, signature: int, e: int, n: int) -> bool
def solve(data: dict) -> dict
```

`solve()` is the blackbox entry point; `sign()` and `verify()` are the underlying primitives
it's built on and are tested directly as well.

#### Input
- `sign(message, d, n)`: `message` is raw `bytes` of any length (including empty); `d` is the
  signer's private exponent; `n` is the RSA modulus (both small, textbook-scale ints).
- `verify(message, signature, e, n)`: `message` as above; `signature` is the claimed signature
  (`int`); `e` is the public exponent; `n` is the modulus that must match the key pair used to
  sign (verifying with a different `n`/`e` pair must fail).
- `solve(data)`: `data` is a `dict` with `"op"` equal to `"sign"` or `"verify"`.
  - `"sign"`: `{"op": "sign", "message": <str>, "d": <int>, "n": <int>}`
  - `"verify"`: `{"op": "verify", "message": <str>, "signature": <int>, "e": <int>, "n": <int>}`
  - `message` arrives as a plain UTF-8 string (encode with `.encode("utf-8")`).

#### Output (`result`)
- `sign(...)` returns an `int` signature in the range `[0, n)`.
- `verify(...)` returns `bool`.
- `solve(...)` returns `{"signature": <int>}` for op `"sign"`, or `{"valid": <bool>}` for op
  `"verify"`.

#### Guarantees / rules
- Pure functions, deterministic, no I/O, no network. Standard library only (`hashlib.sha256`
  may be used as the hash primitive; the RSA sign/verify math itself, reducing the digest mod
  `n` and using modular exponentiation, e.g. Python's built-in `pow(base, exp, mod)`, must be
  implemented directly, not via any external crypto library).
- Must implement "hash, then sign": `signature = (SHA-256(message) mod n)^d mod n`; verification
  recomputes `SHA-256(message) mod n` and compares it to `signature^e mod n`.
- Any single-bit change to `message` must (with overwhelming probability) produce a different
  signature, and verification against the original signature must then fail.
- Verifying a genuine `(message, signature)` pair with an unrelated key pair's public key
  `(e, n)` must return `False`.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

Fill in `solve` below. Run the cell (Shift+Enter) after each change.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
"""Starter code for 5.4.c. Student completes the TODOs.
This file intentionally FAILS the tests until implemented."""
import hashlib


def _digest_mod_n(message: bytes, n: int) -> int:
    """Hash message with SHA-256 and reduce it to an integer mod n.

    This is the 'hash, then sign' step from atom.md: we never run RSA on the raw
    message, only on a fixed-size digest of it.
    """
    # TODO: compute hashlib.sha256(message).digest(), convert it to an int with
    #       int.from_bytes(..., "big"), and reduce it mod n.
    raise NotImplementedError("implement _digest_mod_n() — see the problem statement above")


def sign(message: bytes, d: int, n: int) -> int:
    """Sign `message` with private exponent d and modulus n.

    signature = hash(message)^d mod n
    """
    # TODO: compute the digest with _digest_mod_n(), then return pow(digest, d, n).
    raise NotImplementedError("implement sign() — see the problem statement above")


def verify(message: bytes, signature: int, e: int, n: int) -> bool:
    """Verify that `signature` is a valid signature over `message` under public
    key (e, n).

    Recomputes hash(message) mod n and compares it to signature^e mod n.
    """
    # TODO: recompute the expected digest with _digest_mod_n(), recover the digest
    #       the signer actually signed with pow(signature, e, n), and compare them.
    raise NotImplementedError("implement verify() — see the problem statement above")


def solve(data: dict) -> dict:
    """Blackbox entry point. Dispatches on data['op']:
      - {"op": "sign", "message": <str>, "d": <int>, "n": <int>}
            -> {"signature": <int>}
      - {"op": "verify", "message": <str>, "signature": <int>, "e": <int>, "n": <int>}
            -> {"valid": <bool>}
    Messages are given as UTF-8 strings; d/e/n/signature are given as ints (small,
    textbook-scale RSA parameters — JSON-friendly, no hex/bignum encoding needed).
    """
    # TODO: encode data["message"] as utf-8, then dispatch to sign() or verify()
    #       depending on data["op"], and wrap the result in the right dict shape.
    raise NotImplementedError("implement solve() — see the problem statement above")


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

import hashlib


N = 3233
E = 17
D = 2753

def _oracle_digest_mod_n(message: bytes, n: int) -> int:
    digest_bytes = hashlib.sha256(message).digest()
    return int.from_bytes(digest_bytes, "big") % n


def _oracle_sign(message: bytes, d: int, n: int) -> int:
    return pow(_oracle_digest_mod_n(message, n), d, n)


def _find_d(e: int, phi_n: int) -> int:
    for cand in range(2, phi_n):
        if (e * cand) % phi_n == 1:
            return cand
    raise ValueError("no modular inverse found")


_P2, _Q2 = 59, 61
N2 = _P2 * _Q2
_PHI2 = (_P2 - 1) * (_Q2 - 1)
E2 = 7
D2 = _find_d(E2, _PHI2)


def test_sign_matches_oracle():
    message = b"transfer $100 to acct 42"
    got = impl.sign(message, D, N)
    assert got == _oracle_sign(message, D, N)


def test_valid_signature_verifies_true():
    message = b"pay Bob $50"
    signature = impl.sign(message, D, N)
    assert impl.verify(message, signature, E, N) is True


def test_tampered_message_fails_verification():
    message = b"pay Bob $50"
    tampered = b"pay Bob $950"
    signature = impl.sign(message, D, N)
    assert impl.verify(tampered, signature, E, N) is False


def test_wrong_public_key_fails_verification():
    message = b"deploy build 42"
    signature = impl.sign(message, D, N)
    assert impl.verify(message, signature, E2, N2) is False


def test_solve_sign_op():
    result = impl.solve({"op": "sign", "message": "hello", "d": D, "n": N})
    expected = _oracle_sign(b"hello", D, N)
    assert result == {"signature": expected}


def test_solve_verify_op_valid():
    message = "audit log entry 7"
    signature = _oracle_sign(message.encode("utf-8"), D, N)
    result = impl.solve({"op": "verify", "message": message, "signature": signature, "e": E, "n": N})
    assert result == {"valid": True}


def test_solve_verify_op_tampered_message():
    message = "audit log entry 7"
    signature = _oracle_sign(message.encode("utf-8"), D, N)
    result = impl.solve(
        {"op": "verify", "message": "audit log entry 8", "signature": signature, "e": E, "n": N}
    )
    assert result == {"valid": False}


def test_single_bit_change_in_message_flips_signature():
    a = b"password reset for user 1000"
    b = b"password reset for user 1001"
    sig_a = impl.sign(a, D, N)
    sig_b = impl.sign(b, D, N)
    assert sig_a != sig_b


def test_empty_message_signs_and_verifies():
    message = b""
    signature = impl.sign(message, D, N)
    assert impl.verify(message, signature, E, N) is True


def test_verify_rejects_signature_from_different_message_same_key():
    msg1 = b"approve invoice 9001"
    msg2 = b"approve invoice 9002"
    sig1 = impl.sign(msg1, D, N)
    assert impl.verify(msg2, sig1, E, N) is False


def test_signature_is_deterministic():
    message = b"same input, same output"
    sig1 = impl.sign(message, D, N)
    sig2 = impl.sign(message, D, N)
    assert sig1 == sig2


def test_signature_in_valid_range():
    message = b"range check"
    signature = impl.sign(message, D, N)
    assert 0 <= signature < N


def test_solve_unknown_op_raises():
    import pytest

    with pytest.raises(Exception):
        impl.solve({"op": "bogus", "message": "x"})


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Call `solve` on an input you make up, and predict the answer first.

- [ ] Your prediction matched

## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it